# Synthetic Ellipse Orientation Experiment

Creates synthetic images with ellipses at **known orientations**, runs all four pipeline variants on them, and measures how well each model recovers the ground-truth angle.

**Design**: YOLO is run normally on each image (low detection rate on synthetic data is expected and interesting). For the three SAM2 variants the GT bounding box is used as the prompt — this isolates mask-quality differences from detection differences, which is exactly the variable we want.

| Variant | Prompt source | What we're testing |
|---|---|---|
| YOLOv8 | YOLO (end-to-end) | YOLO mask head quality |
| YOLOv8 + SAM2 zero-shot | GT bbox → SAM2 base | SAM2 base mask quality |
| YOLOv8 + SAM2 fine-tuned | GT bbox → SAM2 ft | Fine-tuned mask quality |
| SAM2-auto | None (dense grid) | Auto mode without bbox constraint |

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")
if "/home/users/cayleigh/BoulderNet/YOLOv8-BeyondEarth/src" in sys.path:
    sys.path.remove("/home/users/cayleigh/BoulderNet/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import pandas as pd
import torch
from pathlib import Path
from shapely.geometry import Polygon
from shapely import segmentize
from scipy.stats import kstest
from tqdm.notebook import tqdm

from sahi import AutoDetectionModel
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

plt.rcParams.update({"font.size": 10, "figure.dpi": 130})
np.random.seed(42)
print("Imports OK")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# YOLO weights: search common Sherlock locations, download from GDrive if missing
_YOLO_CANDIDATES = [
    Path.home() / "tmp" / "YOLOv8BeyondEarth" / "yolov8_model" / "yolov8-m-boulder-detection-tmp.pt",
    Path("/scratch/users/cayleigh/tmp/YOLOv8BeyondEarth/yolov8_model/yolov8-m-boulder-detection-tmp.pt"),
    Path("/home/users/cayleigh/tmp/YOLOv8BeyondEarth/yolov8_model/yolov8-m-boulder-detection-tmp.pt"),
]
YOLO_WEIGHTS = next((p for p in _YOLO_CANDIDATES if p.exists()), None)
if YOLO_WEIGHTS is None:
    import gdown
    _dest = Path.home() / "tmp" / "YOLOv8BeyondEarth" / "yolov8_model"
    _dest.mkdir(parents=True, exist_ok=True)
    YOLO_WEIGHTS = _dest / "yolov8-m-boulder-detection-tmp.pt"
    print("Downloading YOLO weights (~50MB)...")
    gdown.download("https://drive.google.com/uc?id=1DJ3Ek4NI1uEzlB1pyor-KDRN8_pEorVp",
                   str(YOLO_WEIGHTS), quiet=False)
    print("Download complete")

# SAM2 checkpoints: search common locations
_SAM2_BASE_CANDIDATES = [
    Path("/scratch/users/cayleigh/checkpoints/sam2.1_hiera_small.pt"),
    Path("/home/users/cayleigh/checkpoints/sam2.1_hiera_small.pt"),
    Path.home() / "checkpoints" / "sam2.1_hiera_small.pt",
]
SAM2_BASE_CKPT = next((p for p in _SAM2_BASE_CANDIDATES if p.exists()), _SAM2_BASE_CANDIDATES[0])

_SAM2_FT_CANDIDATES = [
    Path("/scratch/users/cayleigh/sam2_finetuned/sam2_boulder_best.pt"),
    Path("/home/users/cayleigh/sam2_finetuned/sam2_boulder_best.pt"),
    Path.home() / "sam2_finetuned" / "sam2_boulder_best.pt",
]
SAM2_FT_CKPT = next((p for p in _SAM2_FT_CANDIDATES if p.exists()), _SAM2_FT_CANDIDATES[0])

SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_s.yaml"
OUT_DIR     = Path.home() / "tmp" / "synthetic_orientation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, p in [("YOLO", YOLO_WEIGHTS), ("SAM2 base", SAM2_BASE_CKPT), ("SAM2 ft", SAM2_FT_CKPT)]:
    status = "found" if p.exists() else "NOT FOUND"
    print(f"  {name:15s}: {status}  ({p})")

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {DEVICE}")

# ── Experiment parameters ─────────────────────────────────────────────────────
N           = 300    # number of synthetic images (one ellipse each)
CANVAS_PX   = 256    # image size (SAM2 upscales to 1024 internally)
A_WORLD     = 15     # semi-major axis in world units
B_WORLD     = 10     # semi-minor axis → AR = 1.5, within paper's 1.2–2.0 filter
PPU         = 6      # pixels per world unit → ellipse is ~180×120 px in 256×256 canvas
BG_SIGMA    = 8      # background noise sigma (mimics LRO NAC dark terrain)
CONF_THRESH = 0.05   # lowered threshold to increase YOLO detection rate on synthetic data
AR_MIN, AR_MAX = 1.2, 2.0
BBOX_MARGIN = 1.1    # GT bbox expansion factor before passing to SAM2

In [ ]:
# ── Load all four models ───────────────────────────────────────────────────────

# YOLO
detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=str(YOLO_WEIGHTS),
    confidence_threshold=CONF_THRESH,
    device=DEVICE,
    image_size=1024,
)
print("YOLO loaded")

# SAM2 zero-shot
sam2_base = build_sam2(SAM2_CONFIG, str(SAM2_BASE_CKPT), device=DEVICE)
predictor_zs = SAM2ImagePredictor(sam2_base)
print("SAM2 zero-shot loaded")

# SAM2 fine-tuned (same architecture, different weights)
sam2_ft = build_sam2(SAM2_CONFIG, str(SAM2_BASE_CKPT), device=DEVICE)
sam2_ft.load_state_dict(torch.load(SAM2_FT_CKPT, map_location=DEVICE))
predictor_ft = SAM2ImagePredictor(sam2_ft)
print("SAM2 fine-tuned loaded")

# SAM2-auto (dense grid of point prompts, no bounding boxes)
sam2_auto_model  = build_sam2(SAM2_CONFIG, str(SAM2_BASE_CKPT), device=DEVICE)
mask_generator   = SAM2AutomaticMaskGenerator(
    sam2_auto_model,
    points_per_side=16,
    pred_iou_thresh=0.7,
    stability_score_thresh=0.85,
)
print("SAM2-auto loaded")

In [ ]:
# ── Synthetic image generation ─────────────────────────────────────────────────

def make_smooth_ellipse(a, b, theta_deg, cx=0.0, cy=0.0, n_pts=200):
    theta = np.radians(theta_deg)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    x = a * np.cos(t) * np.cos(theta) - b * np.sin(t) * np.sin(theta) + cx
    y = a * np.cos(t) * np.sin(theta) + b * np.sin(t) * np.cos(theta) + cy
    return Polygon(np.column_stack([x, y]))


def synth_image(theta_deg, canvas=CANVAS_PX, ppu=PPU, bg_sigma=BG_SIGMA):
    """
    Create a single synthetic grayscale image with one ellipse at theta_deg.
    Returns:
        image_rgb  — uint8 H×W×3 (grayscale replicated, as SAM2 expects)
        gt_poly    — Shapely polygon in pixel coords
        gt_box_px  — [x1, y1, x2, y2] tight bounding box in pixel coords
    """
    # Dark noisy background (like LRO NAC terrain)
    bg = np.random.normal(28, bg_sigma, (canvas, canvas)).clip(0, 255).astype(np.uint8)

    # Ellipse centered in canvas
    origin = canvas // 2
    poly   = make_smooth_ellipse(A_WORLD, B_WORLD, theta_deg)
    pts    = np.array(poly.exterior.coords[:-1])
    pts_px = (pts * ppu + origin).astype(np.int32)  # world → pixel coords
    gt_poly_px = Polygon(pts_px.astype(float))

    # Fill ellipse with bright interior + subtle texture (similar to sunlit rock)
    brightness = int(np.random.uniform(155, 200))
    cv2.fillPoly(bg, [pts_px], brightness)
    yy, xx = np.mgrid[0:canvas, 0:canvas]
    texture = (np.sin(xx * 0.4) * np.cos(yy * 0.4) * 10).astype(np.int16)
    mask_filled = np.zeros((canvas, canvas), np.uint8)
    cv2.fillPoly(mask_filled, [pts_px], 1)
    bg = np.clip(bg.astype(np.int16) + texture * mask_filled, 0, 255).astype(np.uint8)

    # GT bbox with margin
    minx, miny, maxx, maxy = gt_poly_px.bounds
    cx_px, cy_px = (minx + maxx) / 2, (miny + maxy) / 2
    hw = (maxx - minx) / 2 * BBOX_MARGIN
    hh = (maxy - miny) / 2 * BBOX_MARGIN
    gt_box_px = np.array([
        max(0, cx_px - hw), max(0, cy_px - hh),
        min(canvas - 1, cx_px + hw), min(canvas - 1, cy_px + hh)
    ], dtype=np.float32)

    image_rgb = np.stack([bg, bg, bg], axis=-1)
    return image_rgb, gt_poly_px, gt_box_px


# Preview a grid of examples at different orientations and noise seeds
example_angles = [10, 30, 45, 60, 75, 90, 120, 150]
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for col_i, theta in enumerate(example_angles):
    for row_i in range(2):
        img, gt_poly, gt_box = synth_image(theta)
        ax = axes[row_i, col_i]
        ax.imshow(img[:, :, 0], cmap="gray", vmin=0, vmax=255)
        minx, miny, maxx, maxy = gt_box
        rect = mpatches.Rectangle((minx, miny), maxx-minx, maxy-miny,
                                   lw=1, edgecolor="cyan", facecolor="none")
        ax.add_patch(rect)
        if row_i == 0:
            ax.set_title(f"θ={theta}°", fontsize=8)
        ax.axis("off")
plt.suptitle(f"Synthetic ellipses (a={A_WORLD}, b={B_WORLD}, AR=1.5) — two random backgrounds per angle",
             fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "01_synthetic_examples.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Orientation measurement (paper pipeline) ──────────────────────────────────

RES_SYNTH = 1.0 / PPU   # world units per pixel


def measure_orientation(polygon, res=RES_SYNTH):
    """
    Paper pipeline: segmentize → fitEllipse → MRR → boulder_row.
    Returns (angle180, aspect_ratio) or None.
    angle180 is in GEOSPATIAL convention: 0°=North/y-axis, clockwise, in [0,180).
    """
    if polygon is None or polygon.is_empty:
        return None
    if len(polygon.exterior.coords) < 5:
        return None
    try:
        row_seg = pd.Series({"geometry": segmentize(polygon, res)})
        ellipse_poly, _, _, _ = fitEllipse(row_seg)
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        _, _, long_ax, short_ax, _, _, _, angle180 = boulder_row(mrr_row)
        if short_ax < 1e-6:
            return None
        return float(angle180 % 180), float(long_ax / short_ax)
    except Exception:
        return None


def mask_to_polygon(binary_mask):
    """Binary mask → largest contour → Shapely polygon."""
    cnts, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea).squeeze()
    if cnt.ndim != 2 or len(cnt) < 5:
        return None
    return Polygon(cnt.astype(float))


def box_iou(a, b):
    """IoU between two [x1,y1,x2,y2] boxes."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter)


print("Utility functions defined")

# Sanity check: input angles are in geospatial convention; synth_image takes math convention
# convert: math_theta = (90 - geo_angle) % 180
print("Sanity check (input in geo convention, should recover ~same value):")
for test_geo in [15, 45, 75, 105, 135, 165]:
    test_math = (90 - test_geo) % 180
    _, gt_poly, _ = synth_image(test_math)
    r = measure_orientation(gt_poly)
    print(f"  geo θ={test_geo:3d}°  →  recovered {r[0]:.1f}°  AR={r[1]:.2f}" if r else f"  geo θ={test_geo}° → fit failed")

In [ ]:
np.random.seed(42)
# Generate input angles in GEOSPATIAL convention (0°=North/y-axis, CW)
# so they match angle180 from boulder_row directly.
# Internally make_smooth_ellipse takes math convention, so we convert:
#   math_theta = (90 - geo_angle) % 180

input_angles_geo = np.random.uniform(0, 180, N)          # what we'll compare against
input_angles_math = (90 - input_angles_geo) % 180        # what we pass to make_smooth_ellipse

images, gt_polys, gt_boxes = [], [], []
for theta_math in input_angles_math:
    img, poly, box = synth_image(theta_math)
    images.append(img)
    gt_polys.append(poly)
    gt_boxes.append(box)

print(f"Generated {N} synthetic images  (canvas={CANVAS_PX}px, PPU={PPU}, A={A_WORLD}, B={B_WORLD})")

# GT orientation baseline — should now match input_angles_geo
gt_orientations = []
for poly in gt_polys:
    r = measure_orientation(poly)
    gt_orientations.append(r[0] if (r and AR_MIN <= r[1] <= AR_MAX) else None)

n_valid_gt = sum(1 for x in gt_orientations if x is not None)
print(f"GT pipeline: {n_valid_gt}/{N} valid measurements")

# Sanity check
print("\nSanity check (should recover ~input angle):")
for i in [0, 1, 2, 3, 4, 5]:
    geo = input_angles_geo[i]
    rec = gt_orientations[i]
    print(f"  geo={geo:.1f}°  →  recovered {rec:.1f}°" if rec else f"  geo={geo:.1f}°  → fit failed")

In [ ]:
# ── YOLO inference ────────────────────────────────────────────────────────────
# Run YOLO on each image. Match detections to GT by bbox IoU.
# YOLO was trained on real planetary imagery so detection rate on synthetic
# data may be low — that itself is an informative result.

YOLO_MATCH_IOU = 0.20   # low threshold since synthetic images look different from training data

yolo_orientations   = [None] * N
yolo_detected       = [False] * N
yolo_n_detections   = []

for i, (img, gt_box) in enumerate(tqdm(zip(images, gt_boxes), total=N, desc="YOLO")):
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    results = detection_model.model(
        img_bgr, imgsz=1024, verbose=False, device=DEVICE
    )
    result = results[0]
    yolo_n_detections.append(len(result.boxes))

    if result.boxes is None or len(result.boxes) == 0:
        continue
    if result.masks is None:
        continue

    boxes_xyxy = result.boxes.xyxy.cpu().numpy()    # (M, 4)
    confs      = result.boxes.conf.cpu().numpy()    # (M,)
    masks_raw  = result.masks.data.cpu().numpy()    # (M, H, W) float in [0,1]

    # Find the detection that best matches the GT box
    best_iou, best_idx = 0.0, -1
    for j, det_box in enumerate(boxes_xyxy):
        if confs[j] < CONF_THRESH:
            continue
        iou = box_iou(gt_box, det_box)
        if iou > best_iou:
            best_iou, best_idx = iou, j

    if best_iou < YOLO_MATCH_IOU:
        continue

    # Resize YOLO's predicted mask to canvas size and extract polygon
    raw_mask = masks_raw[best_idx]  # float [0,1] at model output resolution
    raw_mask_resized = cv2.resize(raw_mask, (CANVAS_PX, CANVAS_PX), interpolation=cv2.INTER_LINEAR)
    binary_mask = (raw_mask_resized > 0.5).astype(np.uint8) * 255

    pred_poly = mask_to_polygon(binary_mask)
    r = measure_orientation(pred_poly)
    if r and AR_MIN <= r[1] <= AR_MAX:
        yolo_orientations[i] = r[0]
        yolo_detected[i] = True

n_yolo_det  = sum(yolo_detected)
n_yolo_meas = sum(1 for x in yolo_orientations if x is not None)
print(f"YOLO: {n_yolo_det}/{N} images matched GT (IoU>{YOLO_MATCH_IOU}), {n_yolo_meas} valid orientations")
print(f"Mean detections per image: {np.mean(yolo_n_detections):.1f}")

In [ ]:
# ── SAM2 zero-shot and fine-tuned (GT bbox as prompt) ────────────────────────

sam2_zs_orientations = [None] * N
sam2_ft_orientations = [None] * N

for i, (img, gt_box) in enumerate(tqdm(zip(images, gt_boxes), total=N, desc="SAM2 prompted")):
    box_input = gt_box[None]   # shape (1, 4) as SAM2 expects

    with torch.no_grad():
        # Zero-shot
        predictor_zs.set_image(img)
        masks_zs, _, _ = predictor_zs.predict(box=box_input, multimask_output=False)

        # Fine-tuned
        predictor_ft.set_image(img)
        masks_ft, _, _ = predictor_ft.predict(box=box_input, multimask_output=False)

    for masks, store in [(masks_zs, sam2_zs_orientations), (masks_ft, sam2_ft_orientations)]:
        if masks is None or len(masks) == 0:
            continue
        binary = (masks[0, 0] > 0).astype(np.uint8) * 255
        pred_poly = mask_to_polygon(binary)
        r = measure_orientation(pred_poly)
        if r and AR_MIN <= r[1] <= AR_MAX:
            store[i] = r[0]

n_zs = sum(1 for x in sam2_zs_orientations if x is not None)
n_ft = sum(1 for x in sam2_ft_orientations if x is not None)
print(f"SAM2 zero-shot: {n_zs}/{N} valid orientations")
print(f"SAM2 fine-tuned: {n_ft}/{N} valid orientations")

In [ ]:
# ── SAM2-auto (no bbox prompt) ────────────────────────────────────────────────
# Finds the generated mask that best overlaps the GT polygon. If none overlaps
# sufficiently it means SAM2-auto missed the ellipse (also an informative result).

SAM2_AUTO_MIN_IOU = 0.15

sam2_auto_orientations    = [None] * N
sam2_auto_matched         = [False] * N
sam2_auto_n_masks         = []

for i, (img, gt_poly) in enumerate(tqdm(zip(images, gt_polys), total=N, desc="SAM2-auto")):
    with torch.no_grad():
        auto_masks = mask_generator.generate(img)
    sam2_auto_n_masks.append(len(auto_masks))

    if not auto_masks:
        continue

    # Match: find the auto mask with highest IoU vs GT polygon rasterized to binary
    gt_arr = np.zeros((CANVAS_PX, CANVAS_PX), np.uint8)
    gt_pts = np.array(gt_poly.exterior.coords[:-1]).astype(np.int32)
    cv2.fillPoly(gt_arr, [gt_pts], 1)

    best_iou, best_poly = 0.0, None
    for am in auto_masks:
        seg = am["segmentation"].astype(np.uint8)  # bool → uint8
        inter = int((seg & gt_arr).sum())
        union = int((seg | gt_arr).sum())
        iou = inter / union if union > 0 else 0.0
        if iou > best_iou:
            best_iou = iou
            best_poly = mask_to_polygon((seg * 255).astype(np.uint8))

    if best_iou < SAM2_AUTO_MIN_IOU:
        continue

    sam2_auto_matched[i] = True
    r = measure_orientation(best_poly)
    if r and AR_MIN <= r[1] <= AR_MAX:
        sam2_auto_orientations[i] = r[0]

n_auto_match = sum(sam2_auto_matched)
n_auto_meas  = sum(1 for x in sam2_auto_orientations if x is not None)
print(f"SAM2-auto: {n_auto_match}/{N} images matched GT (IoU>{SAM2_AUTO_MIN_IOU}), {n_auto_meas} valid orientations")
print(f"Mean auto masks per image: {np.mean(sam2_auto_n_masks):.1f}")

In [ ]:
df = pd.DataFrame({
    "input_angle":    input_angles_geo,      # geospatial convention, matches angle180
    "gt_pipeline":    gt_orientations,
    "yolo":           yolo_orientations,
    "sam2_zeroshot":  sam2_zs_orientations,
    "sam2_finetuned": sam2_ft_orientations,
    "sam2_auto":      sam2_auto_orientations,
})

df.to_csv(OUT_DIR / "synthetic_orientation_results.csv", index=False)
print("Results saved to", OUT_DIR / "synthetic_orientation_results.csv")
print("\nValid measurements per variant:")
for col in ["gt_pipeline", "yolo", "sam2_zeroshot", "sam2_finetuned", "sam2_auto"]:
    n = df[col].notna().sum()
    print(f"  {col:20s}: {n:4d}/{N}  ({n/N*100:.0f}%)")

In [ ]:
# ── Qualitative: example masks for a single image ─────────────────────────────
# Find an image where all four models produced a valid measurement

fully_covered = df[
    df["yolo"].notna() &
    df["sam2_zeroshot"].notna() &
    df["sam2_finetuned"].notna() &
    df["sam2_auto"].notna()
].index.tolist()

if fully_covered:
    idx = fully_covered[0]
    img_ex   = images[idx]
    theta_ex = input_angles[idx]
    gt_box_ex= gt_boxes[idx]

    # Re-run inference on this one image to get actual masks for display
    img_bgr = cv2.cvtColor(img_ex, cv2.COLOR_RGB2BGR)
    res_yolo = detection_model.model(img_bgr, imgsz=1024, verbose=False, device=DEVICE)[0]

    with torch.no_grad():
        predictor_zs.set_image(img_ex)
        masks_zs_ex, _, _ = predictor_zs.predict(box=gt_box_ex[None], multimask_output=False)
        predictor_ft.set_image(img_ex)
        masks_ft_ex, _, _ = predictor_ft.predict(box=gt_box_ex[None], multimask_output=False)
        auto_masks_ex = mask_generator.generate(img_ex)

    # Reconstruct YOLO binary mask
    yolo_bin_ex = None
    if res_yolo.masks is not None and len(res_yolo.masks) > 0:
        boxes_ex = res_yolo.boxes.xyxy.cpu().numpy()
        best_iou, best_j = 0.0, -1
        for j, b in enumerate(boxes_ex):
            iou = box_iou(gt_box_ex, b)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= YOLO_MATCH_IOU:
            raw = res_yolo.masks.data.cpu().numpy()[best_j]
            raw_r = cv2.resize(raw, (CANVAS_PX, CANVAS_PX))
            yolo_bin_ex = (raw_r > 0.5).astype(np.uint8) * 255

    # SAM2-auto best match
    gt_arr_ex = np.zeros((CANVAS_PX, CANVAS_PX), np.uint8)
    cv2.fillPoly(gt_arr_ex, [gt_boxes[idx].astype(np.int32).reshape(2,2)[[0,1],[0,1]]], 1)  # skip
    gt_pts_ex = np.array(gt_polys[idx].exterior.coords[:-1]).astype(np.int32)
    cv2.fillPoly(gt_arr_ex, [gt_pts_ex], 1)
    auto_bin_ex, best_auto_iou = None, 0.0
    for am in auto_masks_ex:
        seg = am["segmentation"].astype(np.uint8)
        iou = int((seg & gt_arr_ex).sum()) / max(int((seg | gt_arr_ex).sum()), 1)
        if iou > best_auto_iou:
            best_auto_iou = iou
            auto_bin_ex = (seg * 255).astype(np.uint8)

    COLORS = {
        "YOLOv8":          (100, 180, 255),
        "SAM2 zero-shot":  (255, 100, 100),
        "SAM2 fine-tuned": (255, 165,  60),
        "SAM2-auto":       (160, 100, 255),
    }
    panel_masks = [
        ("YOLOv8",          yolo_bin_ex,                          df.at[idx, "yolo"]),
        ("SAM2 zero-shot",  (masks_zs_ex[0,0]>0).astype(np.uint8)*255 if masks_zs_ex is not None else None,
                            df.at[idx, "sam2_zeroshot"]),
        ("SAM2 fine-tuned", (masks_ft_ex[0,0]>0).astype(np.uint8)*255 if masks_ft_ex is not None else None,
                            df.at[idx, "sam2_finetuned"]),
        ("SAM2-auto",       auto_bin_ex,                          df.at[idx, "sam2_auto"]),
    ]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for col_i, (label, bmask, meas_angle) in enumerate(panel_masks):
        color = COLORS[label]
        # Top: mask outline on image
        overlay = img_ex.copy()
        if bmask is not None:
            cnts, _ = cv2.findContours(bmask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            cv2.drawContours(overlay, cnts, -1, color, 2)
        axes[0, col_i].imshow(overlay)
        angle_str = f"θ_meas={meas_angle:.1f}°" if meas_angle is not None else "not detected"
        axes[0, col_i].set_title(f"{label}\n{angle_str}  (true={theta_ex:.1f}°)", fontsize=8)
        axes[0, col_i].axis("off")
        # Bottom: binary mask zoomed
        if bmask is not None:
            axes[1, col_i].imshow(bmask, cmap="gray", interpolation="nearest")
        else:
            axes[1, col_i].text(0.5, 0.5, "no detection", ha="center", va="center",
                                transform=axes[1, col_i].transAxes)
        axes[1, col_i].set_title("Binary mask", fontsize=8)
        axes[1, col_i].axis("off")

    plt.suptitle(f"Example: image #{idx}, true θ={theta_ex:.1f}°", fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "02_example_masks.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No image with all four valid measurements — showing best available")

In [ ]:
# ── Orientation histograms ────────────────────────────────────────────────────

BINS = np.linspace(0, 180, 37)   # 5° bins, matching the paper
CX   = (BINS[:-1] + BINS[1:]) / 2

SERIES = [
    ("input_angle",    "Input angles\n(ground truth)",        "seagreen"),
    ("gt_pipeline",    "GT polygon\n(paper pipeline)",        "mediumseagreen"),
    ("yolo",           "YOLOv8",                              "steelblue"),
    ("sam2_zeroshot",  "SAM2 zero-shot\n(GT bbox prompt)",    "tomato"),
    ("sam2_finetuned", "SAM2 fine-tuned\n(GT bbox prompt)",   "darkorange"),
    ("sam2_auto",      "SAM2-auto\n(no bbox)",                "mediumpurple"),
]

fig, axes = plt.subplots(1, 6, figsize=(24, 3.8))

for ax, (col, title, color) in zip(axes, SERIES):
    angles = df[col].dropna().values
    if len(angles) == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title, fontsize=8.5)
        continue
    counts, _ = np.histogram(angles, bins=BINS)
    ax.bar(CX, counts, width=4.5, color=color, edgecolor="white", lw=0.3)
    ax.axhline(len(angles) / len(CX), color="k", ls="--", lw=0.8, alpha=0.6)
    ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180],
           xlabel="Orientation (°)", ylabel="Count")
    ax.set_title(f"{title}\n(n={len(angles)})", fontsize=8.5)
    ax.spines[["top", "right"]].set_visible(False)
    if col != "input_angle":
        D, p = kstest(angles / 180.0, "uniform")
        ax.text(0.97, 0.97, f"D={D:.3f}\np={p:.3f}",
                transform=ax.transAxes, ha="right", va="top", fontsize=7.5,
                color="red" if p < 0.05 else "gray")

plt.suptitle(
    f"Orientation histograms on synthetic ellipses (N={N}, a={A_WORLD}, b={B_WORLD}, AR=1.5)\n"
    "KS D-stat and p-value vs. uniform. Red = significant departure from flat distribution.",
    fontsize=10
)
plt.tight_layout()
plt.savefig(OUT_DIR / "03_orientation_histograms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 03_orientation_histograms.png")

In [ ]:
# ── Scatter: input angle vs. recovered angle ──────────────────────────────────

SCATTER_SERIES = [
    ("gt_pipeline",    "GT polygon\n(paper pipeline)",     "mediumseagreen"),
    ("yolo",           "YOLOv8",                           "steelblue"),
    ("sam2_zeroshot",  "SAM2 zero-shot",                   "tomato"),
    ("sam2_finetuned", "SAM2 fine-tuned",                  "darkorange"),
    ("sam2_auto",      "SAM2-auto",                        "mediumpurple"),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (col, label, color) in zip(axes, SCATTER_SERIES):
    valid = df[["input_angle", col]].dropna()
    if valid.empty:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(label)
        continue
    ax.scatter(valid["input_angle"], valid[col], s=6, alpha=0.4, color=color, rasterized=True)
    ax.plot([0, 180], [0, 180], "k--", lw=0.8)
    ax.set(xlim=(0, 180), ylim=(0, 180),
           xticks=[0, 45, 90, 135, 180], yticks=[0, 45, 90, 135, 180],
           xlabel="True angle (°)", ylabel="Measured angle (°)")
    ax.set_title(f"{label}\n(n={len(valid)})", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    mae = np.mean(np.abs(valid["input_angle"].values - valid[col].values))
    ax.text(0.05, 0.95, f"MAE={mae:.1f}°", transform=ax.transAxes, fontsize=9, va="top")

plt.suptitle("True vs. measured orientation — diagonal = perfect recovery", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "04_scatter_recovery.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 04_scatter_recovery.png")

In [ ]:
# ── Angle error distribution by model ─────────────────────────────────────────
# For images where both the GT pipeline and a model produced a measurement,
# compare absolute angular error vs. the known input angle.

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))

ERR_SERIES = [
    ("yolo",           "YOLOv8",           "steelblue"),
    ("sam2_zeroshot",  "SAM2 zero-shot",   "tomato"),
    ("sam2_finetuned", "SAM2 fine-tuned",  "darkorange"),
    ("sam2_auto",      "SAM2-auto",        "mediumpurple"),
]

for ax, (col, label, color) in zip(axes, ERR_SERIES):
    valid = df[["input_angle", col]].dropna()
    if valid.empty:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(label)
        continue
    errors = np.abs(valid["input_angle"].values - valid[col].values)
    # Angular error wraps at 180°
    errors = np.minimum(errors, 180 - errors)
    ax.hist(errors, bins=36, range=(0, 90), color=color, edgecolor="white", lw=0.3)
    ax.axvline(np.median(errors), color="k", ls="--", lw=1.2, label=f"median={np.median(errors):.1f}°")
    ax.set(xlabel="|error| (°)", ylabel="Count", xlim=(0, 90))
    ax.set_title(f"{label}\nmae={np.mean(errors):.1f}°  med={np.median(errors):.1f}°", fontsize=9)
    ax.legend(fontsize=7)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Absolute angular error vs. known GT — lower / flatter = better", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "05_error_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 05_error_distributions.png")

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────

rows = []
for col, label in [
    ("gt_pipeline",    "GT polygon (paper pipeline)"),
    ("yolo",           "YOLOv8"),
    ("sam2_zeroshot",  "SAM2 zero-shot (GT bbox)"),
    ("sam2_finetuned", "SAM2 fine-tuned (GT bbox)"),
    ("sam2_auto",      "SAM2-auto"),
]:
    valid = df[["input_angle", col]].dropna()
    if valid.empty:
        rows.append({"Model": label, "n": 0, "MAE (°)": "-", "Median err (°)": "-",
                     "KS D": "-", "KS p": "-"})
        continue
    errs = np.minimum(
        np.abs(valid["input_angle"].values - valid[col].values),
        180 - np.abs(valid["input_angle"].values - valid[col].values)
    )
    D, p = kstest(valid[col].values / 180.0, "uniform")
    rows.append({
        "Model":          label,
        "n":              len(valid),
        "MAE (°)":        f"{np.mean(errs):.1f}",
        "Median err (°)": f"{np.median(errs):.1f}",
        "KS D":           f"{D:.3f}",
        "KS p":           f"{p:.3f}",
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
df_summary.to_csv(OUT_DIR / "summary_table.csv", index=False)